# Test 2 — Public-Benchmark Coref RAG Eval (BGE-M3 local, DAPR + BEIR)

**Research question:** On standard informative-text retrieval, does *coref-before-embed*
and *hybrid sparse+dense* beat a conventional dense baseline?

Unlike Test 1 (one Wikipedia doc, ~20 self-generated pronoun questions), this test uses
**recognized public benchmarks** with their **fixed passages, queries, and official qrels**.
No custom q-gen, no re-chunking, no LLM-as-judge, no paid embedding API — everything runs
**locally on Kaggle GPU** with **BAAI/bge-m3** via FlagEmbedding.

## Three ingestion variants (all BGE-M3 unified)

| Label | Retrieval |
|-------|-----------|
| **baseline** | Dense on the **original** passage text |
| **coref_dense** | Dense on the **LingMess coref rewrite** (same `_id`) |
| **coref_hybrid** | **RRF** fuse: BGE-M3 **sparse(original)** + BGE-M3 **dense(coref)**, `RRF_K=60` |

**Invariants**
- The text **returned to the user** is always the **original** (never the coref rewrite).
- Gold = **official qrels only** (`corpus_id` match, `score > 0`).
- Same passage IDs across all variants; 1:1 coref alignment is asserted.
- BGE-M3 **sparse is a learned lexical** representation (not BM25). Fallback only if OOM.

## Datasets (informative text, not scientific papers)

| Priority | Dataset | Source | Content |
|----------|---------|--------|---------|
| Smoke | **BEIR nfcorpus** | `BeIR/nfcorpus` | Small consumer-health articles (~300 test Qs) |
| Main 1 | **DAPR ConditionalQA** | `UKPLab/dapr` | Wikipedia-style, document-context (271 Qs / 69k passages) |
| Main 2 | **DAPR nq-hard** (optional) | `UKPLab/dapr` | Harder NQ/Wikipedia cases |
| Standard | **BEIR hotpotqa** (optional) | `BeIR/hotpotqa` | Subsampled: gold + candidate pool (5M full is out of scope) |

**Metrics:** Recall@5, nDCG@10, MRR (via `pytrec_eval`), plus flip counts (recovered / hurt / both-fail)
vs the dense baseline. Relative comparison across variants matters more than absolute SOTA numbers.


## 1. Setup & Install

`FlagEmbedding` provides BGE-M3 (dense + learned-sparse in one model). `pytrec_eval` gives the
official TREC metrics. `fastcoref` + pinned `transformers<5` is reused from Test 1 (LingMess needs
the transformers 4.x API).

In [ ]:
!pip install -q -U FlagEmbedding
!pip install -q datasets pytrec_eval tqdm pandas numpy scipy rank_bm25 sentence-transformers
# fastcoref's LingMessModel targets transformers 4.x. Kaggle ships 5.x, which breaks LingMess. Pin.
!pip install -q "transformers<5" fastcoref
!python -m spacy download en_core_web_sm
# IMPORTANT: after this cell finishes, RESTART THE KERNEL once so pinned transformers 4.x is the
# version actually imported, then run the cells below (skip re-running this one).
# NOTE: pip may print dependency-conflict WARNINGS about numba / cudf / cuml (Kaggle's RAPIDS
# packages, unused here). Safe to ignore.

## 2. Config

No API keys required — BGE-M3 runs locally on the Kaggle GPU. Dense + sparse vectors are cached
to disk per `(dataset, variant, passage_id-set)` so the notebook is **resume-friendly**.

In [ ]:
import os, logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("coref_public_eval")

# --- Core config (engineering knobs from the task spec) ---
TOP_K       = 5                 # primary hit cutoff (Recall@5, flips)
RRF_K       = 60                # reciprocal-rank-fusion constant
EMBED_MODEL = "BAAI/bge-m3"
CACHE_DIR   = "./embed_cache"
BATCH_SIZE  = 12                # 8-16; lower if OOM
MAX_LEN     = 8192              # BGE-M3 max sequence length
RUN_DEPTH   = 100               # keep top-100 per query for metric computation

# --- Coref windowing (LingMess Longformer length limit) ---
COREF_WINDOW_WORDS = 350

# --- Optional-dataset subsampling (documented in findings) ---
HOTPOTQA_MAX_CORPUS = 60000     # gold passages + candidate pool cap for the 5M-corpus dataset

os.makedirs(CACHE_DIR, exist_ok=True)
log.info("Config | model=%s | TOP_K=%d | RRF_K=%d | batch=%d | cache=%s",
         EMBED_MODEL, TOP_K, RRF_K, BATCH_SIZE, CACHE_DIR)

## 3. Coreference engine (LingMess, local) — reused from Test 1

Same LingMess setup as Test 1: force **eager** attention (Longformer is incompatible with SDPA on
transformers 5.x), resolve each pronoun mention to the first **named** mention in its cluster, and
apply char-offset edits right-to-left. We reuse `resolve_coref_text` and add
`resolve_sentences_aligned` for document-level, window-budgeted coref that maps rewrites back to the
**passage** boundaries DAPR/BEIR already define (no re-chunking).

In [ ]:
import re, time

SENT_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")
PRONOUN_RE = re.compile(
    r"\b(he|she|him|her|his|hers|they|them|their|theirs|it|its)\b", re.IGNORECASE,
)

def split_sentences(text):
    return [s.strip() for s in SENT_SPLIT_RE.split(text.strip()) if s.strip()]

# LingMess (Longformer) is incompatible with SDPA; transformers 5.x raises instead of falling back.
# Force eager on the config the model init reads. Guard flag on the class prevents double-wrapping.
from transformers.models.auto.modeling_auto import AutoModel

if not getattr(AutoModel, "_coref_eager_patched", False):
    _orig_from_config = AutoModel.from_config.__func__

    @classmethod
    def _from_config_eager(cls, config, **kwargs):
        try:
            config._attn_implementation = "eager"
        except Exception:
            pass
        kwargs.setdefault("attn_implementation", "eager")
        try:
            return _orig_from_config(cls, config, **kwargs)
        except (TypeError, ValueError):
            kwargs.pop("attn_implementation", None)
            return _orig_from_config(cls, config, **kwargs)

    AutoModel.from_config = _from_config_eager
    AutoModel._coref_eager_patched = True

from fastcoref import LingMessCoref

_has_cuda = False
try:
    import torch
    _has_cuda = torch.cuda.is_available()
except Exception:
    pass

coref_model = LingMessCoref(device="cuda" if _has_cuda else "cpu")
log.info("LingMessCoref loaded | device=%s", "cuda" if _has_cuda else "cpu")

PRONOUN_SET = {
    "he", "she", "him", "her", "his", "hers", "they", "them", "their", "theirs",
    "it", "its", "himself", "herself", "themselves", "itself",
}

def _is_pronoun_span(text):
    return text.strip().lower() in PRONOUN_SET

def _has_pronoun_token(text):
    return any(w.strip(".,;:'\"").lower() in PRONOUN_SET for w in text.split())

def _pick_representative(spans_text):
    """Best cluster 'name': prefer proper-noun, pronoun-free spans; else any pronoun-free span;
    else None (never reinject a pronoun). Shortest candidate avoids trailing clauses."""
    proper = [s for s in spans_text
              if not _has_pronoun_token(s) and any(t[:1].isupper() for t in s.split())]
    clean = [s for s in spans_text if not _has_pronoun_token(s)]
    pool = proper or clean
    if not pool:
        return None
    return min(pool, key=len).strip()

def coref_edits(text):
    """Sorted (start, end, replacement) char-offset edits replacing pronouns with cluster names."""
    if not text.strip():
        return []
    preds = coref_model.predict(texts=[text])
    if not preds:                       # LingMess returns [] if text exceeds its length limit
        return []
    clusters = preds[0].get_clusters(as_strings=False)
    edits = []
    for cluster in clusters:
        spans_text = [text[s:e] for s, e in cluster]
        rep = _pick_representative(spans_text)
        if not rep:
            continue
        for (s, e), st in zip(cluster, spans_text):
            if _is_pronoun_span(st) and st.strip().lower() != rep.lower():
                repl = rep + "'s" if st.strip().lower() in {"his", "her", "its", "their", "hers", "theirs"} else rep
                edits.append((s, e, repl))
    return sorted(edits, key=lambda x: x[0])

def apply_edits(text, edits):
    out = text
    for s, e, repl in sorted(edits, key=lambda x: x[0], reverse=True):
        out = out[:s] + repl + out[e:]
    return out

def resolve_coref_text(text):
    """Passage-level coref: replace pronoun mentions with their representative name."""
    return apply_edits(text, coref_edits(text))

### Document-level coref (preferred when `doc_id` is available)

DAPR passages carry a `doc_id` and `paragraph_no`, so we can resolve coref with **document context**
(a pronoun's antecedent may live in an earlier passage) and then map the rewrite back to each
passage. We coref the concatenated passages of a document in word-budgeted windows and route
char-offset edits back to the owning passage — preserving the **official passage boundaries** exactly
(1:1, no re-chunking). Passage-level coref is the fallback when no `doc_id` is present (e.g. BEIR).

In [ ]:
def coref_passages_doc_level(passages, window_words=COREF_WINDOW_WORDS):
    """passages: list of dicts with 'text' in document order (same doc_id).
    Returns list of coref-rewritten texts, 1:1 aligned with input passages."""
    texts = [p["text"] for p in passages]
    out = list(texts)

    # Group passage indices into windows under window_words (whole passages, never split).
    windows, cur, cur_w = [], [], 0
    for i, t in enumerate(texts):
        w = len(t.split())
        if cur and cur_w + w > window_words:
            windows.append(cur); cur, cur_w = [], 0
        cur.append(i); cur_w += w
    if cur:
        windows.append(cur)

    for win in windows:
        spans, pos, parts = [], 0, []
        for k, idx in enumerate(win):
            if k > 0:
                parts.append(" "); pos += 1
            start = pos
            parts.append(texts[idx]); pos += len(texts[idx])
            spans.append((idx, start, pos))
        joined = "".join(parts)
        edits = coref_edits(joined)
        by_p = {}
        for (s, e, repl) in edits:
            for (idx, ss, se) in spans:
                if ss <= s and e <= se:
                    by_p.setdefault(idx, []).append((s - ss, e - ss, repl))
                    break
        for idx, local in by_p.items():
            out[idx] = apply_edits(texts[idx], local)
    return out

## 4. BGE-M3 encoder (local, dense + learned-sparse) with disk cache

One model produces both a **dense** vector and a **learned-sparse lexical** weight dict per text.
We cache both to `CACHE_DIR` keyed by `(dataset, variant, text-role, content-hash)` so re-runs and
resumes skip re-encoding. Query and passage caches are separate.

**Fallback (OOM):** if BGE-M3 fails to load/encode on the GPU, we fall back to
`bge-small-en-v1.5` (dense) + `rank_bm25` (sparse, hybrid only) and log which path was used. The
findings report records the path actually taken.

In [ ]:
import hashlib, pickle
import numpy as np
from tqdm.auto import tqdm

EMBED_PATH = "bge-m3"   # overwritten to "fallback" if BGE-M3 OOMs

def _hash_texts(texts):
    h = hashlib.sha1()
    for t in texts:
        h.update(t.encode("utf-8", "ignore")); h.update(b"\x00")
    return h.hexdigest()[:16]

def _cache_file(dataset, variant, role, texts):
    key = f"{dataset}__{variant}__{role}__{EMBED_PATH}__{_hash_texts(texts)}"
    return os.path.join(CACHE_DIR, key + ".pkl")

# ---- Load BGE-M3 (unified dense + sparse). Fall back to bge-small + BM25 on failure. ----
bge_m3 = None
fallback_dense = None
try:
    from FlagEmbedding import BGEM3FlagModel
    bge_m3 = BGEM3FlagModel(EMBED_MODEL, use_fp16=True)
    log.info("Loaded BGE-M3 (dense+sparse, fp16) on local GPU")
except Exception as e:
    log.warning("BGE-M3 load failed (%s) -> FALLBACK bge-small-en-v1.5 + BM25", e)
    EMBED_PATH = "fallback"
    from sentence_transformers import SentenceTransformer
    fallback_dense = SentenceTransformer("BAAI/bge-small-en-v1.5",
                                         device="cuda" if _has_cuda else "cpu")

def _encode_bge_m3(texts, want_sparse):
    """Return (dense: np.ndarray [n,d] float32 normalized, sparse: list[dict] or None)."""
    dense_out, sparse_out = [], [] if want_sparse else None
    for i in tqdm(range(0, len(texts), BATCH_SIZE), desc="BGE-M3 encode", leave=False):
        batch = texts[i:i + BATCH_SIZE]
        try:
            out = bge_m3.encode(batch, batch_size=len(batch), max_length=MAX_LEN,
                                return_dense=True, return_sparse=want_sparse,
                                return_colbert_vecs=False)
        except RuntimeError as e:            # OOM within a batch -> retry singly
            if "out of memory" not in str(e).lower():
                raise
            import torch; torch.cuda.empty_cache()
            out = {"dense_vecs": [], "lexical_weights": []}
            for t in batch:
                o = bge_m3.encode([t], batch_size=1, max_length=MAX_LEN,
                                  return_dense=True, return_sparse=want_sparse,
                                  return_colbert_vecs=False)
                out["dense_vecs"].append(np.asarray(o["dense_vecs"])[0])
                if want_sparse:
                    out["lexical_weights"].append(o["lexical_weights"][0])
        dv = np.asarray(out["dense_vecs"], dtype="float32")
        dense_out.append(dv)
        if want_sparse:
            lw = out["lexical_weights"]
            sparse_out.extend([{str(k): float(v) for k, v in d.items()} for d in lw])
    dense = np.vstack(dense_out).astype("float32")
    dense /= (np.linalg.norm(dense, axis=1, keepdims=True) + 1e-9)
    return dense, sparse_out

def _encode_fallback(texts, want_sparse):
    dense = fallback_dense.encode(texts, batch_size=BATCH_SIZE, normalize_embeddings=True,
                                  show_progress_bar=True).astype("float32")
    sparse = None                            # BM25 handled separately in hybrid path
    return dense, sparse

def encode_cached(dataset, variant, role, texts, want_sparse=False):
    """Encode with disk cache. role in {'passage','query'}. Returns (dense, sparse|None)."""
    cf = _cache_file(dataset, variant, role + ("_sp" if want_sparse else ""), texts)
    if os.path.exists(cf):
        with open(cf, "rb") as f:
            obj = pickle.load(f)
        log.info("cache HIT  | %s", os.path.basename(cf))
        return obj["dense"], obj.get("sparse")
    log.info("cache MISS | encoding %d %s texts (%s/%s)", len(texts), role, dataset, variant)
    if bge_m3 is not None:
        dense, sparse = _encode_bge_m3(texts, want_sparse)
    else:
        dense, sparse = _encode_fallback(texts, want_sparse)
    with open(cf, "wb") as f:
        pickle.dump({"dense": dense, "sparse": sparse}, f)
    return dense, sparse

## 5. Retrieval: dense, BGE-M3 learned-sparse, and RRF hybrid

- **Dense**: cosine (dot on normalized vectors) — `baseline` and `coref_dense`.
- **Sparse**: BGE-M3 lexical-weight overlap score (`compute_lexical_matching_score`), computed
  over the **original** text — the lexical half of `coref_hybrid`.
- **Hybrid**: **RRF** fuse of `sparse(original)` ranks and `dense(coref)` ranks, `RRF_K=60`.
  When the BM25 fallback is active, sparse ranks come from `rank_bm25` instead.

In [ ]:
from scipy.sparse import csr_matrix

def dense_ranklist(q_dense, p_dense, depth=RUN_DEPTH):
    """Return, per query, a list of (passage_idx, score) sorted desc, top-`depth`."""
    sims = q_dense @ p_dense.T                       # (nq, np)
    order = np.argsort(-sims, axis=1)[:, :depth]
    return [[(int(j), float(sims[qi, j])) for j in order[qi]] for qi in range(sims.shape[0])]

def _sparse_to_matrix(sparse_dicts):
    """Build a token-id -> column csr matrix from list of {tokid: weight}."""
    vocab, rows, cols, vals = {}, [], [], []
    for r, d in enumerate(sparse_dicts):
        for k, v in d.items():
            c = vocab.setdefault(k, len(vocab))
            rows.append(r); cols.append(c); vals.append(v)
    n_cols = max(len(vocab), 1)
    m = csr_matrix((vals, (rows, cols)), shape=(len(sparse_dicts), n_cols), dtype="float32")
    return m, vocab

def sparse_ranklist(q_sparse, p_sparse, depth=RUN_DEPTH):
    """BGE-M3 lexical matching: dot product of aligned token-weight vectors."""
    pm, vocab = _sparse_to_matrix(p_sparse)
    # Align queries to the passage vocabulary.
    rows, cols, vals = [], [], []
    for r, d in enumerate(q_sparse):
        for k, v in d.items():
            if k in vocab:
                rows.append(r); cols.append(vocab[k]); vals.append(v)
    qm = csr_matrix((vals, (rows, cols)), shape=(len(q_sparse), pm.shape[1]), dtype="float32")
    sims = (qm @ pm.T).toarray()                     # (nq, np)
    order = np.argsort(-sims, axis=1)[:, :depth]
    return [[(int(j), float(sims[qi, j])) for j in order[qi]] for qi in range(sims.shape[0])]

def bm25_ranklist(query_texts, passage_texts, depth=RUN_DEPTH):
    """Fallback sparse ranker when BGE-M3 is unavailable."""
    from rank_bm25 import BM25Okapi
    tok = lambda s: re.findall(r"[a-z0-9]+", s.lower())
    bm25 = BM25Okapi([tok(t) for t in passage_texts])
    out = []
    for q in query_texts:
        scores = bm25.get_scores(tok(q))
        order = np.argsort(-scores)[:depth]
        out.append([(int(j), float(scores[j])) for j in order])
    return out

def rrf_fuse(ranklists, k=RRF_K, depth=RUN_DEPTH):
    """Reciprocal-rank fusion of multiple per-query rank lists. ranklists: list over systems,
    each = list over queries of [(pidx, score), ...]. Returns fused [(pidx, rrf_score), ...]."""
    nq = len(ranklists[0])
    fused = []
    for qi in range(nq):
        acc = {}
        for sys_rl in ranklists:
            for rank, (pidx, _s) in enumerate(sys_rl[qi]):
                acc[pidx] = acc.get(pidx, 0.0) + 1.0 / (k + rank + 1)
        ordered = sorted(acc.items(), key=lambda x: -x[1])[:depth]
        fused.append([(int(p), float(s)) for p, s in ordered])
    return fused

## 6. Metrics (official qrels, `pytrec_eval`)

We build a TREC-style `run` (`query_id -> {passage_id: score}`) per variant and score with
`pytrec_eval` against the **official qrels** (unchanged). Reported: **Recall@5**, **nDCG@10**,
**MRR**. A per-query **hit** = any retrieved id (top-`TOP_K`) is in that query's qrels with
`score > 0`. Flip analysis compares each coref variant to the dense baseline.

In [ ]:
import pytrec_eval

def build_run(ranklist, query_ids, passage_ids):
    """ranklist: per-query [(pidx, score)]. Returns run dict for pytrec_eval."""
    run = {}
    for qi, qid in enumerate(query_ids):
        run[qid] = {passage_ids[pidx]: float(score) for pidx, score in ranklist[qi]}
    return run

def eval_run(run, qrels, k=TOP_K):
    """Return dict of aggregate metrics + per-query hit set (top-k)."""
    measures = {"recall_5", "ndcg_cut_10", "recip_rank"}
    ev = pytrec_eval.RelevanceEvaluator(qrels, measures)
    per = ev.evaluate(run)
    recall = float(np.mean([v["recall_5"] for v in per.values()])) if per else 0.0
    ndcg = float(np.mean([v["ndcg_cut_10"] for v in per.values()])) if per else 0.0
    mrr = float(np.mean([v["recip_rank"] for v in per.values()])) if per else 0.0

    # top-k hit set (any gold id in the top-k retrieved)
    gold = {q: {d for d, s in rels.items() if s > 0} for q, rels in qrels.items()}
    hits = set()
    for qid, docs in run.items():
        topk = [d for d, _ in sorted(docs.items(), key=lambda x: -x[1])[:k]]
        if gold.get(qid) and (set(topk) & gold[qid]):
            hits.add(qid)
    return {"recall_at_5": recall, "ndcg_at_10": ndcg, "mrr": mrr, "hits": hits, "n": len(qrels)}

## 7. Dataset loaders (fixed passages, queries, official qrels)

Each loader returns a uniform structure:
```
{ "name", "passages": [{"_id","text","doc_id","order"}], "queries": {qid: text}, "qrels": {qid:{pid:score}} }
```
No custom chunking — we use each benchmark's own passage/chunk units and qrels verbatim. For BEIR
we restrict the corpus to the passages referenced by qrels **plus** a capped candidate pool so it
fits on a single GPU (subsampling documented in findings). DAPR ConditionalQA (69k passages) is
indexed in full.

In [ ]:
from datasets import load_dataset

def _restrict_corpus(passages, qrels, extra_pool, cap):
    """Keep all gold passages + up to `cap` total (gold first, then candidate pool)."""
    gold_ids = {pid for rels in qrels.values() for pid, s in rels.items() if s > 0}
    kept = [p for p in passages if p["_id"] in gold_ids]
    seen = {p["_id"] for p in kept}
    for p in extra_pool:
        if len(kept) >= cap:
            break
        if p["_id"] not in seen:
            kept.append(p); seen.add(p["_id"])
    return kept

def load_beir(name, cap=None):
    """BEIR (BeIR/<name>): corpus + test queries + test qrels via HF."""
    corpus = load_dataset(f"BeIR/{name}", "corpus", split="corpus")
    queries_ds = load_dataset(f"BeIR/{name}", "queries", split="queries")
    qrels_ds = load_dataset(f"BeIR/{name}-qrels", split="test")

    qrels = {}
    for r in qrels_ds:
        if int(r["score"]) > 0:
            qrels.setdefault(str(r["query-id"]), {})[str(r["corpus-id"])] = int(r["score"])
    qtext = {str(q["_id"]): (q["text"] or "") for q in queries_ds}
    queries = {qid: qtext[qid] for qid in qrels if qid in qtext}

    passages = [{"_id": str(c["_id"]),
                 "text": ((c.get("title") or "") + " " + (c.get("text") or "")).strip(),
                 "doc_id": str(c["_id"]), "order": i}
                for i, c in enumerate(corpus)]
    if cap:
        passages = _restrict_corpus(passages, qrels, passages, cap)
    log.info("[%s] passages=%d queries=%d qrels=%d", name, len(passages), len(queries), len(qrels))
    return {"name": name, "passages": passages, "queries": queries, "qrels": qrels}

def load_dapr(dataset_name="ConditionalQA", cap=None):
    """DAPR (UKPLab/dapr): passages carry doc_id + paragraph_no for document-level coref."""
    corpus = load_dataset("UKPLab/dapr", f"{dataset_name}-corpus", split="test")
    queries_ds = load_dataset("UKPLab/dapr", f"{dataset_name}-queries", split="test")
    qrels_ds = load_dataset("UKPLab/dapr", f"{dataset_name}-qrels", split="test")

    qrels = {}
    for r in qrels_ds:
        if int(r["score"]) > 0:
            qrels.setdefault(str(r["query_id"]), {})[str(r["corpus_id"])] = int(r["score"])
    qtext = {str(q["_id"]): (q["text"] or "") for q in queries_ds}
    queries = {qid: qtext[qid] for qid in qrels if qid in qtext}

    passages = [{"_id": str(c["_id"]), "text": (c["text"] or ""),
                 "doc_id": str(c.get("doc_id") or c["_id"]),
                 "order": int(c.get("paragraph_no") or 0)}
                for c in corpus]
    if cap:
        passages = _restrict_corpus(passages, qrels, passages, cap)
    log.info("[DAPR %s] passages=%d queries=%d qrels=%d",
             dataset_name, len(passages), len(queries), len(qrels))
    return {"name": f"dapr_{dataset_name}", "passages": passages,
            "queries": queries, "qrels": qrels}

def load_dapr_nq_hard(cap=HOTPOTQA_MAX_CORPUS):
    """DAPR nq-hard: extended qrels carry query/passage text inline. We index the gold passages
    from the NaturalQuestions corpus plus a capped candidate pool."""
    qrels_ds = load_dataset("UKPLab/dapr", "nq-hard", split="test")
    qrels, queries, inline = {}, {}, {}
    for r in qrels_ds:
        qid, pid = str(r["query_id"]), str(r["corpus_id"])
        if int(r["score"]) > 0:
            qrels.setdefault(qid, {})[pid] = int(r["score"])
        queries[qid] = r.get("query") or ""
        inline[pid] = {"_id": pid, "text": r.get("text") or "",
                       "doc_id": str(r.get("doc_id") or pid), "order": 0}
    passages = list(inline.values())
    log.info("[DAPR nq-hard] passages(inline gold)=%d queries=%d qrels=%d",
             len(passages), len(queries), len(qrels))
    return {"name": "dapr_nq_hard", "passages": passages, "queries": queries, "qrels": qrels}

## 8. Build coref rewrites per dataset (1:1 with passage IDs)

For each dataset we produce a `coref_text` per passage, aligned 1:1 with the original by `_id`.
When `doc_id` groups multiple passages we use **document-level** coref (passages resolved together,
in document order); otherwise **passage-level**. Pronoun counts before/after are logged, and the
1:1 ID alignment is asserted. Results are cached to disk so coref runs once per dataset.

In [ ]:
def build_coref_texts(ds):
    """Add 'coref_text' to each passage. Returns (pron_before, pron_after)."""
    cache = os.path.join(CACHE_DIR, f"coref__{ds['name']}.pkl")
    if os.path.exists(cache):
        with open(cache, "rb") as f:
            mapping = pickle.load(f)
        for p in ds["passages"]:
            p["coref_text"] = mapping.get(p["_id"], p["text"])
        log.info("[%s] coref cache HIT (%d passages)", ds["name"], len(mapping))
    else:
        # Group by doc_id for document-level coref; singletons fall back to passage-level.
        from collections import defaultdict
        groups = defaultdict(list)
        for p in ds["passages"]:
            groups[p["doc_id"]].append(p)
        t0 = time.time()
        for did, plist in tqdm(groups.items(), desc=f"coref {ds['name']}"):
            plist.sort(key=lambda x: x["order"])
            if len(plist) > 1:
                rewrites = coref_passages_doc_level(plist)
                for p, rw in zip(plist, rewrites):
                    p["coref_text"] = rw
            else:
                plist[0]["coref_text"] = resolve_coref_text(plist[0]["text"])
        mapping = {p["_id"]: p["coref_text"] for p in ds["passages"]}
        with open(cache, "wb") as f:
            pickle.dump(mapping, f)
        log.info("[%s] coref built | %.1fs | %d passages", ds["name"], time.time() - t0, len(mapping))

    # Invariants: 1:1 alignment.
    assert all("coref_text" in p for p in ds["passages"]), "coref alignment broken"
    orig_all = " ".join(p["text"] for p in ds["passages"])
    coref_all = " ".join(p["coref_text"] for p in ds["passages"])
    pb, pa = len(PRONOUN_RE.findall(orig_all)), len(PRONOUN_RE.findall(coref_all))
    log.info("[%s] pronouns %d -> %d (%.0f%% reduction)",
             ds["name"], pb, pa, 100 * (1 - pa / max(pb, 1)))
    return pb, pa

## 9. Run the three variants on a dataset

- **baseline**: dense(original) ranks.
- **coref_dense**: dense(coref) ranks.
- **coref_hybrid**: RRF( sparse(original), dense(coref) ).

Queries are encoded once (dense + sparse). Passages are encoded per variant (dense original, dense
coref, sparse original) — all cached. Returns metrics + runs for flip analysis.

In [ ]:
def run_dataset(ds):
    name = ds["name"]
    passages = ds["passages"]
    p_ids = [p["_id"] for p in passages]
    q_ids = list(ds["queries"].keys())
    q_texts = [ds["queries"][q] for q in q_ids]
    orig_texts = [p["text"] for p in passages]
    coref_texts = [p["coref_text"] for p in passages]

    want_sparse = (bge_m3 is not None)   # BGE-M3 gives learned sparse; else BM25 fallback

    # ---- Queries ----
    q_dense, q_sparse = encode_cached(name, "query", "query", q_texts, want_sparse=want_sparse)
    # ---- Passages ----
    p_dense_orig,  p_sparse_orig = encode_cached(name, "baseline", "passage", orig_texts,
                                                 want_sparse=want_sparse)
    p_dense_coref, _             = encode_cached(name, "coref_dense", "passage", coref_texts,
                                                 want_sparse=False)

    # ---- Rank lists ----
    rl_baseline = dense_ranklist(q_dense, p_dense_orig)
    rl_corefden = dense_ranklist(q_dense, p_dense_coref)
    if want_sparse:
        rl_sparse = sparse_ranklist(q_sparse, p_sparse_orig)
    else:
        rl_sparse = bm25_ranklist(q_texts, orig_texts)      # fallback path
    rl_hybrid = rrf_fuse([rl_sparse, rl_corefden])          # sparse(orig) + dense(coref)

    variants = {
        "baseline":     rl_baseline,
        "coref_dense":  rl_corefden,
        "coref_hybrid": rl_hybrid,
    }
    results, runs = {}, {}
    for label, rl in variants.items():
        run = build_run(rl, q_ids, p_ids)
        runs[label] = run
        results[label] = eval_run(run, ds["qrels"])
        r = results[label]
        log.info("[%s/%s] R@5=%.3f nDCG@10=%.3f MRR=%.3f",
                 name, label, r["recall_at_5"], r["ndcg_at_10"], r["mrr"])
    return {"name": name, "results": results, "runs": runs,
            "n_passages": len(passages), "n_queries": len(q_ids)}

## 10. Results tables + flip analysis

`summarize(run_out)` produces the per-dataset table (baseline / coref_dense / coref_hybrid with
Δ vs baseline) and the flip counts (recovered / hurt / both-fail) for each coref variant relative
to the dense baseline.

In [ ]:
import pandas as pd

def summarize(run_out):
    res = run_out["results"]
    base = res["baseline"]
    rows = []
    for label in ["baseline", "coref_dense", "coref_hybrid"]:
        r = res[label]
        rows.append({
            "variant": label,
            "recall@5": round(r["recall_at_5"], 4),
            "nDCG@10": round(r["ndcg_at_10"], 4),
            "MRR": round(r["mrr"], 4),
            "Δrecall@5": round(r["recall_at_5"] - base["recall_at_5"], 4),
            "ΔnDCG@10": round(r["ndcg_at_10"] - base["ndcg_at_10"], 4),
        })
    df = pd.DataFrame(rows)
    print(f"\n=== {run_out['name']} | passages={run_out['n_passages']} queries={run_out['n_queries']} ===")
    print(df.to_string(index=False))
    return df

def flips(run_out):
    res = run_out["results"]
    base_hits = res["baseline"]["hits"]
    all_qids = set(run_out["runs"]["baseline"].keys())   # every query with a run
    out = {}
    for label in ["coref_dense", "coref_hybrid"]:
        h = res[label]["hits"]
        recovered = h - base_hits            # baseline miss -> variant hit
        hurt = base_hits - h                 # baseline hit -> variant miss
        both_fail = all_qids - base_hits - h
        out[label] = {"recovered": len(recovered), "hurt": len(hurt),
                      "both_fail": len(both_fail),
                      "recovered_ids": list(recovered)[:10], "hurt_ids": list(hurt)[:10]}
        print(f"\n[{run_out['name']}] {label} vs baseline: "
              f"recovered={len(recovered)} hurt={len(hurt)} both_fail={len(both_fail)}")
    return out

## 11. Smoke run — BEIR nfcorpus (all 3 variants)

Small consumer-health corpus (~3.6k passages, ~300 test queries). Target runtime < 2 h with cache.
Runs the full pipeline end-to-end to validate BGE-M3 local encoding + metrics before the larger DAPR run.

In [ ]:
ALL_RUNS = {}   # name -> run_out, collected for the findings report
COREF_STATS = {}  # name -> (pron_before, pron_after)

ds_nf = load_beir("nfcorpus")            # small; index in full
COREF_STATS["nfcorpus"] = build_coref_texts(ds_nf)
run_nf = run_dataset(ds_nf)
ALL_RUNS["nfcorpus"] = run_nf
df_nf = summarize(run_nf)
flips_nf = flips(run_nf)

## 12. Main run — DAPR ConditionalQA (all 3 variants)

Wikipedia-style informative text with document context (271 queries, 69,199 passages). This is the
benchmark designed to reward document-context understanding (coreference), so it is the core test of
the research question. Full corpus indexed; design targets cache + resume (~2-5 h local).

In [ ]:
ds_cqa = load_dapr("ConditionalQA")
COREF_STATS["dapr_ConditionalQA"] = build_coref_texts(ds_cqa)
run_cqa = run_dataset(ds_cqa)
ALL_RUNS["dapr_ConditionalQA"] = run_cqa
df_cqa = summarize(run_cqa)
flips_cqa = flips(run_cqa)

## 13. Optional runs — DAPR nq-hard & BEIR hotpotqa (subsampled)

Optional per the task spec. `nq-hard` uses the inline gold passages; `hotpotqa` is subsampled to
gold + a candidate pool (`HOTPOTQA_MAX_CORPUS`) since the full 5M corpus is out of scope. Comment
these out to save runtime.

In [ ]:
RUN_OPTIONAL = False   # flip to True to include the optional datasets

if RUN_OPTIONAL:
    ds_nqh = load_dapr_nq_hard()
    COREF_STATS["dapr_nq_hard"] = build_coref_texts(ds_nqh)
    run_nqh = run_dataset(ds_nqh)
    ALL_RUNS["dapr_nq_hard"] = run_nqh
    summarize(run_nqh); flips(run_nqh)

    ds_hp = load_beir("hotpotqa", cap=HOTPOTQA_MAX_CORPUS)
    COREF_STATS["hotpotqa"] = build_coref_texts(ds_hp)
    run_hp = run_dataset(ds_hp)
    ALL_RUNS["hotpotqa"] = run_hp
    summarize(run_hp); flips(run_hp)

## 14. Worked example — a document-context recovery case

Find a query where `coref_dense` or `coref_hybrid` recovered a baseline miss, and show the gold
passage's **original** text (returned to the user) vs its **coref rewrite** (embedded). This is the
DAPR "the venue" -> "the Half Moon, Putney" pattern: the answer passage refers to the entity by a
reference noun, and coref injects the name into the embedded text.

In [ ]:
def show_worked_example(ds, run_out, variant="coref_dense"):
    res = run_out["results"]
    recovered = res[variant]["hits"] - res["baseline"]["hits"]
    if not recovered:
        print(f"No recovery for {variant} on {run_out['name']}."); return
    qid = sorted(recovered)[0]
    gold_ids = {pid for pid, s in ds["qrels"][qid].items() if s > 0}
    by_id = {p["_id"]: p for p in ds["passages"]}
    print("QUERY:", ds["queries"][qid])
    print("\nRecovered by:", variant)
    for gid in list(gold_ids)[:1]:
        p = by_id.get(gid)
        if not p:
            continue
        print("\n--- Gold passage", gid, "---")
        print("RETURNED (original):", p["text"][:500])
        print("\nEMBEDDED (coref)   :", p["coref_text"][:500])
        print("\npronouns  orig:", len(PRONOUN_RE.findall(p["text"])),
              "| coref:", len(PRONOUN_RE.findall(p["coref_text"])))

# Prefer the main dataset for the worked example; fall back to nfcorpus.
for _name in ["dapr_ConditionalQA", "nfcorpus"]:
    if _name in ALL_RUNS:
        _ds = ds_cqa if _name == "dapr_ConditionalQA" else ds_nf
        show_worked_example(_ds, ALL_RUNS[_name]); break

## 15. Write `test-2-findings.md`

Assembles setup, per-dataset result tables, Δ vs baseline, flip counts, one worked example, and
scope notes into the findings file. Re-run after all datasets complete for a full report.

In [ ]:
def _md_table(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |",
             "| " + " | ".join("---" for _ in cols) + " |"]
    for _, r in df.iterrows():
        lines.append("| " + " | ".join(str(r[c]) for c in cols) + " |")
    return "\n".join(lines)

def write_findings(path="test-2-findings.md"):
    path_used = "BGE-M3 (dense + learned-sparse), local Kaggle GPU, fp16" \
        if bge_m3 is not None else "FALLBACK: bge-small-en-v1.5 dense + rank_bm25 sparse"
    out = []
    out.append("# Test 2 — Public-Benchmark Coref RAG Eval (BGE-M3 local)\n")
    out.append(f"**Run date:** {time.strftime('%Y-%m-%d')}  ")
    out.append("**Notebook:** `coref_public_eval.ipynb`  ")
    out.append("**Question:** On standard informative-text retrieval, does coref-before-embed "
               "and hybrid sparse+dense beat a conventional dense baseline?\n")
    out.append("## Setup\n")
    out.append(f"- **Embedding model / path:** {path_used}")
    out.append(f"- **Variants:** baseline (dense original) / coref_dense (dense LingMess rewrite) / "
               f"coref_hybrid (RRF: BGE-M3 sparse(original) + dense(coref), RRF_K={RRF_K})")
    out.append("- **Coref:** fastcoref LingMess, document-level where `doc_id` available, else passage-level")
    out.append(f"- **Metrics:** Recall@5, nDCG@10, MRR (pytrec_eval); TOP_K={TOP_K}")
    out.append("- **Gold:** official qrels only (`score > 0`); passage IDs identical across variants (1:1 coref alignment asserted)")
    out.append("- **Note:** BGE-M3 sparse is a *learned lexical* representation, not BM25.\n")

    for name, run_out in ALL_RUNS.items():
        df = summarize(run_out)
        out.append(f"## {name}\n")
        out.append(f"Passages: {run_out['n_passages']} | Queries: {run_out['n_queries']}")
        pb, pa = COREF_STATS.get(name, (0, 0))
        if pb:
            out.append(f"Pronouns before→after coref: {pb} → {pa} "
                       f"({100*(1-pa/max(pb,1)):.0f}% reduction)\n")
        out.append(_md_table(df) + "\n")
        # flips
        base_hits = run_out["results"]["baseline"]["hits"]
        all_qids = set(run_out["runs"]["baseline"].keys())
        out.append("**Flips vs baseline**\n")
        out.append("| variant | recovered | hurt | both_fail |")
        out.append("| --- | --- | --- | --- |")
        for label in ["coref_dense", "coref_hybrid"]:
            h = run_out["results"][label]["hits"]
            out.append(f"| {label} | {len(h - base_hits)} | {len(base_hits - h)} | "
                       f"{len(all_qids - base_hits - h)} |")
        out.append("")

    out.append("## Scope notes\n")
    out.append("- **DAPR** = document-context / informative Wikipedia-style retrieval.")
    out.append("- **BEIR** = standard comparability (nfcorpus smoke; hotpotqa subsampled to gold + candidate pool).")
    out.append("- **Hybrid** = BGE-M3 sparse(original) + dense(coref) via RRF — *not* BM25 (unless fallback path was used).")
    out.append("- Relative comparison across variants matters more than absolute SOTA numbers.")
    out.append("- No LLM-as-judge, no answer generation, no custom q-gen, no re-chunking of public corpora.")

    text = "\n".join(out) + "\n"
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    log.info("Wrote %s (%d chars)", path, len(text))
    print(text[:1500])

write_findings()